## SHAP & business cutoff

Explains the final LightGBM from `03_lightgbm` and turns its scores into an approval decision.
The model is loaded from MLflow, not retrained. Same data preparation and train/test split as `02_baseline` and `03_lightgbm`.
SHAP shows which features drive the risk. The cutoff analysis translates the ranking into approval rate and defaults avoided.

**1 - Install SHAP and LightGBM**

In [0]:
%pip install shap lightgbm
dbutils.library.restartPython()

**2 - Imports and constants**

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
import shap
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

PATH = "/Volumes/workspace/risco_credito/raw/cs-training.csv"
TARGET = "SeriousDlqin2yrs"
PAST_DUE = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate",
]
RANDOM_STATE = 42

RUN_ID = "bff55fc3fdbb441bad37b58239f23ae8"  # lightgbm_final from 03_lightgbm
EXPECTED_TEST_PR_AUC = 0.4094

**3 - Read CSV, split and load the model from MLflow**

In [0]:
df = pd.read_csv(PATH, index_col=0)

df["past_due_sentinel"] = df[PAST_DUE].isin([96, 98]).any(axis=1).astype(int)
df["income_missing"] = df["MonthlyIncome"].isna().astype(int)
df["dependents_missing"] = df["NumberOfDependents"].isna().astype(int)
df.loc[df["age"] == 0, "age"] = np.nan
df[PAST_DUE] = df[PAST_DUE].where(~df[PAST_DUE].isin([96, 98]))

X = df.drop(columns=TARGET)
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

model = mlflow.lightgbm.load_model(f"runs:/{RUN_ID}/model")
proba_test = model.predict_proba(X_test)[:, 1]

pr_auc = average_precision_score(y_test, proba_test)
print(f"roc_auc {roc_auc_score(y_test, proba_test):.4f} | pr_auc {pr_auc:.4f}")

# Same model, same data -> same score. If not, something changed upstream.
assert abs(pr_auc - EXPECTED_TEST_PR_AUC) < 1e-4, "Loaded model does not reproduce 03_lightgbm"

**4 - Compute SHAP values and check additivity**

In [0]:
explainer = shap.TreeExplainer(model)
sv = explainer(X_test)

# Some shap versions return one column per class; keep class 1 (default)
if sv.values.ndim == 3:
    sv = sv[:, :, 1]

print(f"shap values shape: {sv.values.shape}")

# Additivity: base value + sum of contributions = model raw score (log-odds)
raw = model.predict(X_test, raw_score=True)
reconstructed = sv.base_values + sv.values.sum(axis=1)
print(f"max additivity error: {np.abs(raw - reconstructed).max():.2e}")

base = np.atleast_1d(sv.base_values)[0]
print(f"base value: {base:.3f} log-odds -> {1 / (1 + np.exp(-base)):.2%} | train default rate {y_train.mean():.2%}")

**5 - Global importance and beeswarm**

In [0]:
importance = (
    pd.Series(np.abs(sv.values).mean(axis=0), index=X_test.columns)
    .sort_values(ascending=False)
    .rename("mean_abs_shap")
)
display(importance.round(3).to_frame())

shap.plots.bar(sv, max_display=13, show=False)
plt.title("Global importance (mean |SHAP|, log-odds)")
plt.tight_layout()
plt.show()

shap.plots.beeswarm(sv, max_display=13, show=False)
plt.title("SHAP values per borrower (test set)")
plt.tight_layout()
plt.show()

**6 - Predicted vs. actual default rate for missing-value groups**

In [0]:
check = pd.DataFrame({
    "actual": y_test.values,
    "predicted": proba_test,
    "past_due_sentinel": X_test["past_due_sentinel"].values,
    "income_missing": X_test["income_missing"].values,
})

for flag in ["past_due_sentinel", "income_missing"]:
    summary = check.groupby(flag).agg(
        n=("actual", "size"),
        actual=("actual", "mean"),
        predicted=("predicted", "mean"),
    )
    print(f"\n{flag}")
    print(summary.round(3))

**7 - DebtRatio dependence plot, split by income_missing**